In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================
# CONFIG
# =========================
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

PROCESSED_CONTAINER = "processed-p1"
CURATED_CONTAINER = "curated-p1"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

ARTICLES_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/articles_clean/"
CUSTOMERS_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customers_clean/"
TRANSACTIONS_IN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_clean/"

INTERACTIONS_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_article_interactions/"
CUSTOMER_FEATURES_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
ARTICLE_FEATURES_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/article_features/"
RECOMMENDATION_BASE_OUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/recommendation_dataset/"

# =========================
# LOAD CLEAN DATA
# =========================
articles_df = spark.read.parquet(ARTICLES_IN)
customers_df = spark.read.parquet(CUSTOMERS_IN)
transactions_df = spark.read.parquet(TRANSACTIONS_IN)

# =========================
# BUILD INTERACTIONS
# =========================
interactions = (
    transactions_df.alias("t")
    .join(customers_df.alias("c"), on="customer_id", how="left")
    .join(articles_df.alias("a"), on="article_id", how="left")
)

latest_date_row = transactions_df.select(F.max("t_dat").alias("max_date")).first()
latest_date = latest_date_row["max_date"]
print("Latest transaction date:", latest_date)

# =========================
# CUSTOMER FEATURES
# =========================
customer_features = (
    interactions.groupBy("customer_id")
    .agg(
        F.count("*").alias("customer_purchase_count"),
        F.sum("price").alias("customer_total_spend"),
        F.avg("price").alias("customer_avg_price"),
        F.countDistinct("article_id").alias("customer_unique_articles"),
        F.countDistinct("product_type_name").alias("customer_unique_product_types"),
        F.countDistinct("department_name").alias("customer_unique_departments"),
        F.max("t_dat").alias("customer_last_purchase_date"),
        F.min("t_dat").alias("customer_first_purchase_date")
    )
    .withColumn(
        "customer_recency_days",
        F.datediff(F.lit(latest_date), F.col("customer_last_purchase_date"))
    )
    .withColumn(
        "customer_purchase_frequency",
        F.when(
            F.datediff(F.col("customer_last_purchase_date"), F.col("customer_first_purchase_date")) > 0,
            F.col("customer_purchase_count") / F.datediff(F.col("customer_last_purchase_date"), F.col("customer_first_purchase_date"))
        ).otherwise(None)
    )
)

dept_pref = (
    interactions.groupBy("customer_id", "department_name")
    .count()
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.desc("count"), F.asc("department_name"))
        )
    )
    .filter("rn = 1")
    .select("customer_id", F.col("department_name").alias("customer_preferred_department"))
)

index_pref = (
    interactions.groupBy("customer_id", "index_group_name")
    .count()
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.desc("count"), F.asc("index_group_name"))
        )
    )
    .filter("rn = 1")
    .select("customer_id", F.col("index_group_name").alias("customer_preferred_index_group"))
)

basket_sizes = (
    interactions.groupBy("customer_id", "t_dat")
    .agg(F.countDistinct("article_id").alias("basket_size"))
    .groupBy("customer_id")
    .agg(F.avg("basket_size").alias("customer_avg_basket_size"))
)

customer_features = (
    customer_features
    .join(dept_pref, on="customer_id", how="left")
    .join(index_pref, on="customer_id", how="left")
    .join(basket_sizes, on="customer_id", how="left")
)

# =========================
# ARTICLE FEATURES
# =========================
article_features = (
    interactions.groupBy("article_id")
    .agg(
        F.count("*").alias("article_total_sales"),
        F.countDistinct("customer_id").alias("article_unique_customers"),
        F.avg("price").alias("article_avg_price"),
        F.max("t_dat").alias("article_last_sold_date"),
        F.expr(f"sum(case when t_dat >= date_sub(date('{latest_date}'), 7) then 1 else 0 end)").alias("article_sales_last_7d"),
        F.expr(f"sum(case when t_dat >= date_sub(date('{latest_date}'), 30) then 1 else 0 end)").alias("article_sales_last_30d"),
        F.first("product_type_name", ignorenulls=True).alias("article_product_type_name"),
        F.first("department_name", ignorenulls=True).alias("article_department_name"),
        F.first("index_group_name", ignorenulls=True).alias("article_index_group_name"),
        F.first("colour_group_name", ignorenulls=True).alias("article_colour_group_name"),
        F.first("garment_group_name", ignorenulls=True).alias("article_garment_group_name")
    )
    .withColumn(
        "article_popularity_rank",
        F.dense_rank().over(Window.orderBy(F.desc("article_total_sales")))
    )
)

# =========================
# RECOMMENDATION DATASET
# =========================
recommendation_dataset = (
    interactions
    .join(customer_features, on="customer_id", how="left")
    .join(article_features, on="article_id", how="left")
    .withColumn(
        "same_preferred_department",
        F.when(
            F.col("department_name") == F.col("customer_preferred_department"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "same_preferred_index_group",
        F.when(
            F.col("index_group_name") == F.col("customer_preferred_index_group"),
            1
        ).otherwise(0)
    )
    .withColumn("price_vs_customer_avg", F.col("price") - F.col("customer_avg_price"))
    .withColumn("is_online_channel", F.when(F.col("sales_channel_id") == 2, 1).otherwise(0))
)

# =========================
# WRITE CURATED DATA
# =========================
interactions.write.mode("overwrite").parquet(INTERACTIONS_OUT)
customer_features.write.mode("overwrite").parquet(CUSTOMER_FEATURES_OUT)
article_features.write.mode("overwrite").parquet(ARTICLE_FEATURES_OUT)
recommendation_dataset.write.mode("overwrite").parquet(RECOMMENDATION_BASE_OUT)

print("Curated datasets written successfully.")
print("customer_features:", customer_features.count())
print("article_features:", article_features.count())
print("recommendation_dataset:", recommendation_dataset.count())

Latest transaction date: 2020-09-22


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Curated datasets written successfully.
customer_features: 1362281


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


article_features: 104547
recommendation_dataset: 28813419


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
